# 0.0 Установка # Обновление пакета geomml 

In [ ]:
from pathlib import Path
import os

root = Path(os.getcwd()).resolve().parents[1]  # вверх на 2 уровня
!python {str(root / "scripts" / "bootstrap.py")}

%load_ext autoreload
%autoreload 2

from hydra import initialize_config_dir, compose
from hydra.utils import instantiate

config_name = "default"

with initialize_config_dir(config_dir=str(root / "configs"), version_base="1.3"):
    cfg = compose(config_name=config_name)

# При начальной установке и запуске может не сработать, нужно перезапустить cell. Бага в разработке, пока не исправлена. 


# 0.1 Импорт необходимых модулей # библиотек

In [ ]:
#!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
#!pip install torch-sparse  -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
#!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
#!pip install torch-geometric

import torch.nn.functional as F
from torch_geometric.loader import DataLoader

from geomml.models import build as build_model
from geomml.datasets import build as build_dataset
from geomml.utils.loader import build as build_loaders
from geomml.utils.plot_history import *
from geomml.trainers.multi_task import MultiTaskTrainer

%matplotlib inline

# 0.2 Скачивание датасета Alchemy

In [ ]:
import geomml.datasets.alchemy
import pandas as pd

# !python {str(root / "scripts" / "download_alchemy.py")}

DATA_DIR = Path("data") / "alchemy"
targets = pd.read_csv(DATA_DIR / "final_version.csv")
print(list(targets.columns))
print(targets.head())
print(targets.columns)

# 0.3 Подготовка **QM9+Alchemy** multi-dataset

In [ ]:
#!python {str(root / "scripts" / "preprocess_alchemy.py")}

alchemy = build_dataset("alchemy", normalize=True)
qm9 = build_dataset("qm9",normalize=True)
dataset = build_dataset("unified", datasets=[qm9, alchemy])

print("Alchemy:\t",     alchemy[0])
print("QM9:\t\t",       qm9[0])
print("Full Dataset:\t",dataset[0])

# 1. Постановка задачи

полностью правильный pipeline под GeomML:
RDKit featurization
edge construction (kNN / radius graph)
batching-ready format
совместимость с BaseGraphDataset
кеширование .pt

Просто скажи:

"сделай полный pipeline под GeomML"




1. готовую GNN модель под это
SchNet-like
DimeNet-lite
EGNN (лучший вариант)

2. full training script (training loop production)
AMP (mixed precision) training
gradient clipping
EMA weights
LR warmup
cosine warmup
uncertainty weighting (log_vars)
normalization targets
radius cutoff tuning (3.5–6.0)
training loop (EMA + scheduler + AMP + clipping)


GradNorm (ещё мощнее, но сложнее)
Идея:

балансируем по градиентам, а не по loss

Минус:
нужен hook на backward
усложняет trainer
не всегда стабилен в EGNN

👉 Я бы использовал только если uncertainty weighting не хватает

🚀 Рекомендация (реально практическая)

Для твоего EGNN я бы сделал так:

🔥 Stage 1 (обязательно)
SmoothL1Loss
task weights
🔥 Stage 2 (лучший баланс)
Uncertainty weighting (log_vars)
❌ GradNorm
только если ты делаешь SOTA-подгонку
💡 Если хочешь следующий шаг апгрейда (очень влияет на качество)

Я могу улучшить тебе ещё 3 вещи, которые сильнее loss’а:

🔬 нормализация таргетов QM9 (ОЧЕНЬ важно)
⚡ улучшение radius_graph (ускорение ×3–5)
🧠 residual EGNN + edge features (большой буст качества)

# 2. EGNN Model

EGNN примерно уровня SchNet/PaiNN-lite, совместимая с вашим пайплайном (QM9, Alchemy, UnifiedDataset, BaseModel, MultiTaskTrainer).

пробовать разные архитектуры (EGNN, SchNet, PaiNN, GNN и т.д.)

(EGNN + RBF + AtomEncoder(z + charge + aromatic) + EdgeEncoder + Angle-aware Block + Layer Attention + Encoder + MolecularEGNN + multitask heads + loss)

Да. С учётом того, что у вас уже есть собственный фреймворк (`BaseModel`, `Registry`, multitask, UnifiedDataset), я бы **не стал делать очередную "EGNNLayer + MLP"**, а построил архитектуру ближе к современным работам (PaiNN/GemNet/GraphGPS), но без их огромной сложности.

Я предлагаю такую архитектуру.

```
Atom Embedding
        │
Bond Embedding (optional edge_attr)
        │
RBF Expansion(dist)
        │
────────────────────────────────────────────
EGNN Block × L
    ├── Edge Message
    ├── Edge Gate
    ├── Coordinate Update
    ├── Angle-aware Block
    ├── Residual
    └── LayerNorm
────────────────────────────────────────────
        │
Attention over Layers
        │
Global Mean Pool
        │
+ TDA
+ Task embedding
        │
Fusion
        │
Backbone
        │
Multi-task heads
```

---

# Что будет внутри

## 1. AtomEncoder

Вместо одного

```python
Embedding(z)
```

лучше

```python
Embedding(z)
+
Embedding(charge)
+
Embedding(aromatic)
```

Если этих полей нет — автоматически использовать нули.

---

## 2. Bond Encoder

Если есть

```
edge_attr
```

то

```
bond embedding
```

иначе просто использовать RBF расстояния.

---

## 3. RBF Expansion

Вместо

```
distance
```

использовать

```
RBF(distance)
```

например 32 радиальных функций.

Это намного информативнее.

---

# 4. Новый EGNN Block

Вместо

```
EdgeMLP

↓

NodeMLP

↓

CoordMLP
```

получится

```
distance

↓

RBF

↓

Edge MLP

↓

Edge Gate (sigmoid)

↓

Message

↓

Node Update

↓

Coordinate Update

↓

Angle-aware Block

↓

LayerNorm

↓

Residual
```

---

## 5. Angle-aware Block

Это главный новый компонент.

Идея:

Для каждого ребра

```
i → j
```

смотрим

```
k → i
```

и вычисляем

```
cos θ
```

```
         k

        /

       /

      i -------- j
```

```
θ = ∠kij
```

После этого

```
edge_feature

+

cosθ

↓

MLP

↓

angle message
```

который добавляется в узел.

Это дешёвая версия идеи DimeNet.

Без сферических гармоник.

Без triplet embedding.

Но работает намного лучше обычной EGNN.

---

# 6. Coordinate update

Использовать оригинальную формулу EGNN

```
Δx_i

=

Σ

(x_i−x_j)

/

||x_i−x_j||

·φ(mij)
```

а не

```
shift = rel * scale
```

как сейчас.

---

# 7. Layer Attention

После каждого блока сохраняем

```
h¹

h²

h³

h⁴

...
```

Потом

```
Attention

↓

weighted sum

↓

graph embedding
```

Это лучше, чем использовать только последний слой.

---

# 8. Pooling

Можно оставить

```
global_mean_pool
```

или

```
mean + max
```

что ещё лучше.

---

# 9. Fusion

```
Geometry

+

TDA

+

Task

↓

Fusion MLP
```

оставить практически как есть.

---

# 10. Backbone

Я бы сделал

```
Linear

↓

SiLU

↓

LayerNorm

↓

Dropout

↓

Linear

↓

SiLU

↓

LayerNorm
```

---

# Что получится

```
                z
                │
        Atom Embedding
                │
         h0 ∈ R^128

                │
         edge_index
                │
        distance → RBF
                │
────────────────────────────────────────────

        EGNN Block 1
           │
        Angle Block
           │
        LayerNorm
           │
        Residual
           │
           ▼

        EGNN Block 2
           │
        Angle Block
           │
        LayerNorm
           │
        Residual

           ▼

        EGNN Block 3

           ▼

        EGNN Block 4

────────────────────────────────────────────

        Layer Attention

────────────────────────────────────────────

       Global Mean Pool

────────────────────────────────────────────

      + Task Embedding
      + TDA Embedding

────────────────────────────────────────────

          Fusion

────────────────────────────────────────────

         Backbone

────────────────────────────────────────────

     y
     dipole
     polar
```

## Что такая архитектура даёт

По сравнению с вашей текущей моделью она добавляет:

* использование готового `edge_index` вместо построения графа на каждом `forward`;
* более выразительное кодирование расстояний через RBF;
* управляемую агрегацию сообщений с помощью edge gate;
* корректное обновление координат по оригинальной формуле EGNN;
* учёт углов (`Angle-aware Block`), что особенно важно для молекул, где свойства зависят не только от длин связей, но и от их взаимного расположения;
* attention по выходам всех EGNN-слоёв, а не использование только последнего слоя;
* лучшую стабильность обучения за счёт residual-соединений и нормализации.

Такой вариант остаётся значительно проще, чем полноценные GemNet или Equiformer, но по выразительности заметно превосходит базовую EGNN и хорошо подходит для вашего мультизадачного пайплайна на QM9 и Alchemy.



Здесь модель должна получать два источника информации:
1. GeomML (или EGNN/SchNet/GeoGNN) — геометрия молекулы. Геометрический энкодер --- ```GeomEncoder -> (B,D)```.
2. TDA-признаки (persistent homology) — топологические признаки молекулы.  TDA encoder - обычный MLP.

GCN (плохая)
SchNet-like
DimeNet-lite
EGNN (лучший вариант)
DimeNet++


КЛЮЧЕВАЯ АРХИТЕКТУРА (то что ты хотел)
         QM9 --------\
                      \
                       EGNN Encoder
                      /
Alchemy -------------/

         ↓
   task embedding
         ↓
     fusion layer
         ↓
   shared latent
         ↓
   multi-head output



То есть это уже двухветочная сеть:


```mermaid
flowchart TB

%% =====================================================
%% Molecular Geometry Branch
%% =====================================================
subgraph Geometry_Branch["Geometry Branch (EGNN)"]

A[Atomic Features<br/>Z + Charge + Aromatic + 3D Coordinates]

B[Atom Encoder<br/>Embedding : z, charge, aromatic<br/>h0 ∈ ℝ¹²⁸]

C[Edge Construction<br/>edge_index + optional edge_attr]

D[RBF Distance Expansion: <br/> r_i-r_j -> radial basis features]

E[EGNN Block × L<br/>Edge Message + Edge Gate<br/>Node Update + Coordinate Update]

F[Angle-aware Block<br/>Triplet Geometry<br/>cos θijk Interaction]

G[Residual + LayerNorm]

H[Layer Attention<br/>Weighted aggregation of EGNN layers]

I[Global Mean Pooling<br/>Graph Embedding<br/>h_geom ∈ ℝ¹²⁸]

A --> B --> E
C --> D --> E
D --> E
E --> F --> G --> H --> I

end


%% =====================================================
%% Optional TDA Branch
%% =====================================================
subgraph TDA_Branch["Topology Branch"]

J[Persistent Diagrams / TDA Features]

K[TDA Encoder<br/>MLP]

L[Topological Embedding<br/>h_tda ∈ ℝ¹²⁸]

J --> K --> L

end


%% =====================================================
%% Task Embedding
%% =====================================================
subgraph Task_Branch["Task Conditioning"]

M[Task ID<br/>QM9 / Alchemy / Property]

N[Task Embedding<br/>h_task ∈ ℝ¹²⁸]

M --> N

end

%% =====================================================
%% Fusion
%% =====================================================
I --> O[Feature Concatenation<br/>h_geom + h_tda + h_task]

L --> O
N --> O

O --> P[Fusion MLP<br/>Linear + SiLU + LayerNorm]

P --> Q[Shared Backbone<br/>Residual MLP]


%% =====================================================
%% Prediction Heads
%% =====================================================
subgraph Outputs["Multi Task Outputs"]

R[Energy Head<br/>E ∈ ℝ]

S[Force Prediction<br/>F = -∂E/∂R]

T[Gap / HOMO-LUMO Head<br/>y ∈ ℝ]

U[Dipole Head<br/>μ Regression]

V[Polarizability Head<br/>α Regression]

end


Q --> R --> S

Q --> T
Q --> U
Q --> V

```

| компонент     | у тебя | EGNN v2 |
| ------------- | ------ | ------- |
| node features | ✔      | ✔       |
| distance      | ✔      | ✔       |
| edge MLP      | ✔      | ✔       |
| coord update  | ❌      | ✔       |
| equivariance  | ❌      | ✔       |


Положительные моменты данной реализации EGNN (v2):
1. rotational invariance 
2. translational invariance 
3. learns geometry itself




Получается структура:

```
                 z, charge, aromatic, pos
                           |
                    Atom Encoder
                           |
                    EGNN Blocks
                           |
        +------------------+----------------+
        |                                   |
 Angle-aware Block                 Coordinate Update
        |                                   |
        +------------------+----------------+
                           |
                   Layer Attention
                           |
                  Global Mean Pool
                           |
                 h_geom (128)


h_geom + h_tda + h_task
             |
        Fusion MLP
             |
       Shared Backbone
             |
  +----------+-----------+------------+
  |          |           |            |
Energy      Force       y        Dipole/Polar
```

Для вашей текущей реализации `egnn_v3` это отражает реальные модули:

* `AtomEncoder`
* `EdgeEncoder`
* `EGNNBlock`
* `AngleAwareBlock`
* `LayerAttention`
* `EGNNEncoder`
* `TDAEncoder`
* `Fusion`
* `Backbone`
* `energy_head`
* multitask heads.


поддержка:

* `y`
* `dipole`
* `polar`
* energy prediction
* force prediction через `-∂E/∂R`
* `task_id`
* `tda`
* `mask_*`
* совместимость с вашим `MultiTaskTrainer`



### Важное замечание по вашему датасету

Сейчас у вас вывод:

```
Alchemy:
Data(edge_index=[2, 52], pos=[21,3], z=[21], y=[1,1], task_id=0)

QM9:
Data(edge_index=[2,8], pos=[5,3], z=[5], y=[1,1], task_id=0)
```

То есть пока **нет**:

```
charge
aromatic
edge_attr
```

Эта модель уже это поддерживает:

```python
getattr(batch,"charge",None)
getattr(batch,"aromatic",None)
getattr(batch,"edge_attr",None)
```

Поэтому она не упадёт.

Но чтобы реально использовать Angle-aware + bond information, желательно, чтобы после `AddMolecularFeatures` стало:

```
Data(
 z=[N],
 pos=[N,3],
 edge_index=[2,E],
 edge_attr=[E,F],
 charge=[N],
 aromatic=[N],
 y=[...]
)
```

---

Есть один потенциальный момент: в `AngleAwareBlock` из части 1 сейчас используется двойной цикл по соседям:

```python
for i in range(n):
    for j in range(i+1,n):
```

Для QM9 это нормально, но для больших молекул Alchemy может стать узким местом. Следующим шагом я бы заменил его на **vectorized triplet angle computation через scatter**, что даст ускорение примерно на порядок.


Да. Теперь, когда в датасете уже есть `edge_index`, нет смысла каждый `forward()` заново вызывать `radius_graph()`. Можно построить более современную EGNN, которая использует готовый граф связей и при этом остается эквивариантной.

Я бы переписал модель следующим образом.

### Что изменится

По сравнению с текущей версией:

* ✅ использовать `edge_index` из датасета;
* ✅ добавить residual connections;
* ✅ добавить LayerNorm;
* ✅ заменить расстояние на RBF-базис (значительно лучше одного числа `dist`);
* ✅ добавить edge gate;
* ✅ добавить attention между слоями EGNN;
* ✅ сделать обновление координат более стабильным;
* ✅ оставить совместимость с вашим multitask pipeline (`y`, `dipole`, `polar`, `tda`, `task_id`);
* ✅ оставить возможность позже добавить `edge_attr`, `charge`, `aromatic` без изменения архитектуры.

---

## Архитектура

```
Atom embedding
        │
        ▼
 EGNN Block × L
        │
        ├── residual
        ├── LayerNorm
        ├── edge gate
        ├── coordinate update
        └── graph attention
        │
        ▼
global mean pool
        │
        ├── TDA encoder
        ├── task embedding
        ▼
fusion
        ▼
shared backbone
        ▼
multi-task heads
```

---

### Почему это лучше

Ваша старая версия строила сообщение

```
[h_i , h_j , distance]
```

Это довольно слабое описание геометрии.

Лучше использовать

```
[h_i,
 h_j,
 RBF(distance)]
```

как делают SchNet, PaiNN, GemNet и другие современные модели.

---

Кроме того,

вместо

```python
agg.index_add_(...)
```

лучше использовать

```
edge gate
```

чтобы сеть сама решала, насколько важна каждая связь.

---

### Attention между слоями

Вместо простого

```
layer1
↓

layer2
↓

layer3
↓

layer4
```

лучше

```
layer1
↓

layer2
↓

layer3
↓

layer4
   │
attention
   │
weighted sum
```

Это напоминает подходы в DeeperGCN и GraphGPS и позволяет использовать информацию с разных глубин.

---

### Edge graph

Теперь

```python
edge_index
```

не строится каждый раз.

В encoder:

```python
def forward(
    self,
    z,
    pos,
    edge_index,
    batch,
):
```

и дальше

```python
for layer in self.layers:
    h, pos = layer(
        h,
        pos,
        edge_index,
    )
```

---

### Координаты

Вместо

```python
shift = rel * coord_scale
```

лучше

```
shift =
(rel / (distance + eps))
*
coord_scale
```

как в оригинальном EGNN.

Это делает обновления независимыми от масштаба координат.

---

### Backbone

Я бы сделал

```
Linear
↓

SiLU
↓

LayerNorm
↓

Dropout

↓

Linear
↓

SiLU
↓

LayerNorm
```

а не просто два Linear подряд.

---

### Loss

Оставил бы практически такой же.

Единственное —

сделал бы маску универсальной:

```
loss = Σ task_loss
```

где

```
if hasattr(batch,"mask_y")
```

используется автоматически.

---

## Что получится по уровню

По сравнению с вашей первой EGNN:

| Модель       | Качество |
| ------------ | -------- |
| текущая EGNN | ⭐⭐⭐      |
| новая EGNN   | ⭐⭐⭐⭐     |
| SchNet       | ⭐⭐⭐⭐     |
| PaiNN        | ⭐⭐⭐⭐⭐    |
| GemNet       | ⭐⭐⭐⭐⭐    |
| Equiformer   | ⭐⭐⭐⭐⭐⭐   |

То есть новая версия будет уже заметно ближе к современным геометрическим GNN, оставаясь существенно проще PaiNN и GemNet.

---

**Я бы ещё добавил один компонент** — **Angle-aware block** поверх EGNN (мы обсуждали его раньше). Он использует углы между соседними рёбрами без перехода к полноценной DimeNet-подобной архитектуре. Это даст дополнительную информацию о локальной геометрии практически без серьёзного увеличения числа параметров и хорошо сочетается с уже имеющимся `edge_index`. Это, на мой взгляд, будет наиболее полезным следующим шагом после перехода на использование готового графа связей.
